<a href="https://colab.research.google.com/github/Shorovpaul/Data_Testing/blob/main/Creditcard%5B01%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("Hi")

Hi


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
pip install pandas numpy scikit-learn imbalanced-learn shap xgboost


In [ ]:
import pandas as pd

data = pd.read_csv(r"creditcard.csv")

print(data.shape)
print(data['Class'].value_counts())

(7973, 31)
Class
0.0    7947
1.0      25
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

# Drop rows where 'Class' is NaN before splitting
data_cleaned = data.dropna(subset=['Class'])

X = data_cleaned.drop("Class", axis=1)
y = data_cleaned["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(y_train_bal.value_counts())

Class
0.0    6357
1.0    6357
Name: count, dtype: int64


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_bal = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
import xgboost as xgb

models = {
    "LR": LogisticRegression(max_iter=1000),
    "RF": RandomForestClassifier(n_estimators=200),
    "GB": GradientBoostingClassifier(),
    "KNN": KNeighborsClassifier(),
    "SVM": SVC(probability=True),
    "XGB": xgb.XGBClassifier(eval_metric="logloss")
}

trained_models = {}

for name, model in models.items():
    model.fit(X_train_bal, y_train_bal)
    trained_models[name] = model

NameError: name 'X_train_bal' is not defined

In [ ]:
from sklearn.metrics import average_precision_score

model_scores = {}

for name, model in trained_models.items():
    probs = model.predict_proba(X_test_scaled)[:,1]
    score = average_precision_score(y_test, probs)
    model_scores[name] = score

print(model_scores)

{'LR': np.float64(0.9666666666666667), 'RF': np.float64(1.0), 'GB': np.float64(1.0), 'KNN': np.float64(0.8006269592476489), 'SVM': np.float64(0.9666666666666667), 'XGB': np.float64(1.0)}


In [ ]:
selected_models = {
    name:model for name,model in trained_models.items()
    if model_scores[name] > 0.70
}

NameError: name 'trained_models' is not defined

In [ ]:
import shap
import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import xgboost as xgb

explanations = {}

# Use a smaller sample of X_train_bal as the background dataset for KernelExplainer
# This is crucial for performance with KernelExplainer
background_data_kernel = shap.sample(X_train_bal, 100)

for name, model in selected_models.items():
    if isinstance(model, (RandomForestClassifier, GradientBoostingClassifier, xgb.XGBClassifier)):
        # For tree-based models, use TreeExplainer
        explainer = shap.TreeExplainer(model)
        # TreeExplainer's shap_values method takes the data directly.
        # For classification, it returns a list of arrays (one for each class).
        shap_values = explainer.shap_values(X_train_bal[:1000])

        # Get importance for the positive class (index 1)
        if isinstance(shap_values, list) and len(shap_values) > 1:
            importance = np.abs(shap_values[1]).mean(axis=0)
        else:
            importance = np.abs(shap_values).mean(axis=0)
    else:
        # For other models (LogisticRegression, KNN, SVM), use KernelExplainer
        # KernelExplainer needs a callable prediction function (predict_proba) and a background dataset.
        explainer = shap.KernelExplainer(model.predict_proba, background_data_kernel)
        # Explain a smaller subset of the training data for faster computation with KernelExplainer
        shap_values = explainer.shap_values(X_train_bal[:100]) # Explaining 100 samples

        # Get importance for the positive class (index 1)
        if isinstance(shap_values, list) and len(shap_values) > 1:
            importance = np.abs(shap_values[1]).mean(axis=0)
        else:
            importance = np.abs(shap_values).mean(axis=0)

    explanations[name] = importance

print(explanations)

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

{'LR': array([[3.90710535e-04, 3.90710535e-04],
       [1.16972056e-02, 1.16972056e-02],
       [3.91290060e-02, 3.91290060e-02],
       [0.00000000e+00, 0.00000000e+00],
       [7.36434552e-02, 7.36434552e-02],
       [1.24612318e-04, 1.24612318e-04],
       [1.93560413e-02, 1.93560413e-02],
       [0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00],
       [2.12725871e-02, 2.12725871e-02],
       [5.18231880e-02, 5.18231880e-02],
       [5.53051069e-04, 5.53051069e-04],
       [1.22337088e-01, 1.22337088e-01],
       [2.55092758e-04, 2.55092758e-04],
       [7.13120811e-02, 7.13120811e-02],
       [1.52918566e-02, 1.52918566e-02],
       [1.13250470e-02, 1.13250470e-02],
       [6.80955638e-02, 6.80955638e-02],
       [5.49468548e-04, 5.49468548e-04],
       [5.47671577e-04, 5.47671577e-04],
       [5.50353714e-04, 5.50353714e-04],
       [6.44304424e-04, 6.44304424e-04],
       [7.31757625e-05, 7.31757625e-05],
       [1.18455335e-03, 1.18455335e-03],
       [1

In [ ]:
from sklearn.model_selection import KFold
import shap
import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

kf = KFold(n_splits=5)

stability_scores = {}

for name, model_instance in selected_models.items():

    fold_importance = []

    for train_idx, val_idx in kf.split(X_train_bal):
        # Refit the model for the current fold
        model_instance.fit(X_train_bal[train_idx], y_train_bal.iloc[train_idx])

        # Prepare background data for KernelExplainer from the current fold's training data
        if not isinstance(model_instance, (RandomForestClassifier, GradientBoostingClassifier, xgb.XGBClassifier)):
            num_samples_for_background = min(50, len(X_train_bal[train_idx]))
            current_fold_background = shap.sample(X_train_bal[train_idx], num_samples_for_background)
        else:
            current_fold_background = None # Not needed for TreeExplainer

        # Prepare evaluation data for SHAP, ensuring enough samples are available
        num_samples_to_explain_tree = min(200, len(X_train_bal[val_idx])) # For TreeExplainer
        num_samples_to_explain_kernel = min(50, len(X_train_bal[val_idx])) # For KernelExplainer

        if isinstance(model_instance, (RandomForestClassifier, GradientBoostingClassifier, xgb.XGBClassifier)):
            explainer = shap.TreeExplainer(model_instance)
            # Explain a subset of validation data
            shap_values_fold = explainer.shap_values(X_train_bal[val_idx][:num_samples_to_explain_tree])

            if isinstance(shap_values_fold, list) and len(shap_values_fold) > 1:
                # For classification, usually shap_values[1] refers to the positive class
                importance = np.abs(shap_values_fold[1]).mean(axis=0)
            else:
                importance = np.abs(shap_values_fold).mean(axis=0)
        else:
            explainer = shap.KernelExplainer(model_instance.predict_proba, current_fold_background)
            # Explain a smaller subset for KernelExplainer for performance
            shap_values_fold = explainer.shap_values(X_train_bal[val_idx][:num_samples_to_explain_kernel])

            if isinstance(shap_values_fold, list) and len(shap_values_fold) > 1:
                # For classification, shap_values[1] refers to the positive class
                importance = np.abs(shap_values_fold[1]).mean(axis=0)
            else:
                importance = np.abs(shap_values_fold).mean(axis=0)

        fold_importance.append(importance)

    fold_importance = np.array(fold_importance) # Shape: (n_folds, n_features)

    # Calculate stability: higher stability for lower variance
    # The variance is calculated across all importance values across folds and features.
    stability = 1 / (1 + fold_importance.var())

    stability_scores[name] = stability

print(stability_scores)

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

...

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

{'LR': np.float64(0.9988791752603169), 'RF': np.float64(0.9988400223099897), 'GB': np.float64(0.3842837377214924), 'KNN': np.float64(0.999602594171916), 'SVM': np.float64(0.9994154176538054), 'XGB': np.float32(0.58288807)}


In [ ]:
import numpy as np

final_scores = {}

for name in selected_models:

    performance = model_scores.get(name, 0)
    stability = stability_scores.get(name, 0)

    diversity_values = [
        v for k, v in diversity_scores.items()
        if isinstance(k, tuple) and name in k
    ]

    diversity = np.mean(diversity_values) if diversity_values else 0

    score = (
        0.5 * performance +
        0.3 * stability +
        0.2 * diversity
    )

    final_scores[name] = score

print(final_scores)

NameError: name 'selected_models' is not defined

In [ ]:
final_scores = {}

for name in selected_models:

    performance = model_scores[name]
    stability = stability_scores[name]

    diversity = np.mean([
        v for k,v in diversity_scores.items() if name in k
    ])

    score = (
        0.5 * performance +
        0.3 * stability +
        0.2 * diversity
    )

    final_scores[name] = score

print(final_scores)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

diversity_scores = {}
model_names = list(explanations.keys())

for i in range(len(model_names)):
    for j in range(i + 1, len(model_names)):
        name1 = model_names[i]
        name2 = model_names[j]

        # Ensure the explanation arrays are 1D for cosine similarity
        importance1 = explanations[name1].flatten()
        importance2 = explanations[name2].flatten()

        # Reshape to 2D arrays for cosine_similarity
        importance1_reshaped = importance1.reshape(1, -1)
        importance2_reshaped = importance2.reshape(1, -1)

        # Calculate cosine similarity. Higher similarity means lower diversity.
        similarity = cosine_similarity(importance1_reshaped, importance2_reshaped)[0][0]
        diversity = 1 - similarity # Diversity is inverse of similarity

        diversity_scores[(name1, name2)] = diversity
        diversity_scores[(name2, name1)] = diversity # Store for both orders for easier lookup

print(diversity_scores)

In [ ]:
import numpy as np

final_scores = {}

for name in selected_models:

    performance = model_scores.get(name, 0)
    stability = stability_scores.get(name, 0)

    diversity_values = [
        v for k, v in diversity_scores.items()
        if isinstance(k, tuple) and name in k
    ]

    diversity = np.mean(diversity_values) if diversity_values else 0

    score = (
        0.5 * performance +
        0.3 * stability +
        0.2 * diversity
    )

    final_scores[name] = score

print(final_scores)

In [ ]:
top_models = sorted(final_scores, key=final_scores.get, reverse=True)[:3]

In [ ]:
from sklearn.ensemble import StackingClassifier

estimators = [(name, selected_models[name]) for name in top_models]

meta_model = LogisticRegression()

stack = StackingClassifier(
    estimators=estimators,
    final_estimator=meta_model
)

stack.fit(X_train_bal, y_train_bal)